# Lab 03 — Vibe Code an Agent with a Structured Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-03-vibe-code-an-agent-with-a-structured-workflow/lab-03-vibe-code-an-agent-with-a-structured-workflow.ipynb)

**Topic:** 2 — Vibe Coding for Multi-Agent Systems

**Objective:** Apply the structured vibe coding workflow — specify, scaffold, generate, run, verify, refine

Use an AI coding agent to build a working application without hand-writing the implementation. **The specification you write is the real deliverable; the agent produces the code.**

Full step-by-step instructions are in the Learner Guide.


> **This lab is mostly done in your terminal, not in this notebook.** The work is directing a coding agent (Claude Code, Gemini CLI or Cursor) in a local Git repository. This notebook holds the artifacts — the specification and the acceptance-test runner — so you can read them here and copy them into your project.

In the repo folder those artifacts are `SPEC.md`, `.gitignore`, `documents/mrt.txt` and `verify.py`.


In [ ]:
!pip install -q openai python-dotenv


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Git first, before any AI writes a line

Every change the agent makes then becomes a reviewable diff, and `git checkout .` is your undo button when a generation goes wrong. Run this in your terminal:

```bash
mkdir lab-03-research-agent && cd lab-03-research-agent
git init
claude
```


## 2. Write SPEC.md first

The highest-value step in the lab. Write it yourself — do not ask the agent to write its own specification. Note what makes it useful: concrete file paths, an exact command, and a pass/fail test with a *negative* case.


In [ ]:
SPEC = """# SPEC - Research Assistant Agent

## Goal
A command-line agent that answers a research question by searching a local
document collection and citing which documents it used.

## Inputs
- `python research_agent.py "<question>"` - the question as one CLI argument.
- `documents/` - a folder of `.txt` files forming the searchable collection.

## Outputs
- A prose answer to stdout, under 200 words.
- A "Sources:" line listing the filenames actually used.
- Exit code 0 on success, 1 on any error.

## Constraints
- Python 3.11+, standard library plus `openai` and `python-dotenv` only.
- No web access - the agent may only read files under `documents/`.
- The API key is read from `OPENAI_API_KEY` via python-dotenv. Never hard-coded.
- The agent loop is capped at 5 turns.
- If no document is relevant, say so explicitly rather than inventing an answer.

## Acceptance test
Given `documents/mrt.txt` containing "The Thomson-East Coast Line opened in 2020":

    python research_agent.py "When did the Thomson-East Coast Line open?"

MUST print an answer containing "2020" and a "Sources:" line naming `mrt.txt`.

Given a question with no relevant document:

    python research_agent.py "What is the capital of Peru?"

MUST state that no relevant document was found, and MUST NOT answer "Lima".
"""

with open("SPEC.md", "w") as fh:
    fh.write(SPEC)
print(SPEC)


## 3. Create the fixture the acceptance test needs


In [ ]:
import os

os.makedirs("documents", exist_ok=True)
with open("documents/mrt.txt", "w") as fh:
    fh.write(
        "The Thomson-East Coast Line opened in 2020. It is Singapore's sixth MRT "
        "line and was built in stages, with Stage 1 entering passenger service in "
        "January 2020.\n"
    )
print(open("documents/mrt.txt").read())


## 4. Direct the agent, one component at a time

Scaffold before logic. Prompts to give your coding agent, in order:

> Read SPEC.md. Create only the project skeleton: `research_agent.py` with function stubs and docstrings matching the spec, `requirements.txt`, a `documents/` folder containing the `mrt.txt` fixture from the acceptance test, and a `.gitignore` that excludes `.env` and `__pycache__`. Do not implement any function bodies yet — leave `raise NotImplementedError`.

> Implement `search_documents(query)` only. It reads every `.txt` file under `documents/`, scores each by keyword overlap with the query, and returns the top 3 as a list of `{"filename": str, "content": str}`. Do not touch any other function.

> Now implement `run_agent(question)` — the loop from SPEC.md, capped at 5 turns, exposing `search_documents` as a function tool.

Review each diff with `git diff` before accepting. When it fails, paste the whole traceback back verbatim — concrete errors produce concrete fixes.


## 5. Verify against the acceptance test

`verify.py` in the repo folder runs **both** cases, including the negative one. An agent that passes the positive case and fails the negative one is a hallucinating agent, which is exactly what the negative case exists to catch.

Once your coding agent has produced `research_agent.py` in this folder, run:


In [ ]:
# Requires research_agent.py, produced by your coding agent from SPEC.md.
# Download verify.py from the lab folder, or run this in your local project.
import os

if os.path.exists("research_agent.py") and os.path.exists("verify.py"):
    !python verify.py
else:
    print("Generate research_agent.py with your coding agent first, then re-run.")
    print("verify.py lives in the lab folder in the course repo.")


## What you learned

- The specification is the deliverable; writing `SPEC.md` before any code is what makes the agent's output verifiable.
- An acceptance test written in advance — including a negative case — is the difference between "it ran" and "it works".
- Scaffold before logic, and one component per generation, so every diff stays reviewable.
- Pasting real error output produces far better fixes than describing the symptom.
- Git is the control surface for AI-generated code: small commits, readable diffs, and a reliable undo.
